In [0]:
%pip install -qU databricks-langchain
# %pip install -U langchain-openai

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    # endpoint="databricks-dbrx-instruct",
    endpoint="databricks-claude-3-7-sonnet",
    temperature=0.1,
    max_tokens=256,
    # See https://python.langchain.com/api_reference/community/chat_models/langchain_community.chat_models.databricks.ChatDatabricks.html for other supported parameters
)

In [0]:
from dataclasses import dataclass

from langchain_community.utilities import SQLDatabase


# define context structure to support dependency injection
@dataclass
class RuntimeContext:
    db: SQLDatabase

In [0]:
import os
dbfs_path = "/dbfs/FileStore/Chinook.dbb"
local_path = "/tmp/Chinook.dbb"

# Copy only if not already present
import shutil
if not os.path.exists(local_path):
    dbutils.fs.cp('dbfs:/FileStore/Chinook.dbb', 'file:///tmp/Chinook.dbb')

from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///tmp/Chinook.dbb")

In [0]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///dbfs/FileStore/Chinook.dbb")


In [0]:
usable = db.get_usable_table_names()
print (usable)

In [0]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [0]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [0]:
from langchain.agents import create_agent

agent = create_agent(
    model=chat_model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [0]:
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [0]:
question = "Which table has the largest number of entries?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()